# Overriding train_step, and the backend-agnostic alternative

How to change what one training step does — and why compute_loss() is usually the better place to do it.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 7 — A Deep Dive on Keras](../../../course-web-slides/ch07/index.html) &nbsp;·&nbsp; **Section:** 03 — Writing your own training logic

---

## The escape hatches, in order of how much they cost you

| What you override | What you keep | What it costs |
|---|---|---|
| `compute_loss()` | everything | nothing — **backend agnostic** |
| `train_step()` | callbacks, metrics, progress | the code is backend-specific |
| the whole loop | nothing | everything |

Chapter 17's VAE and diffusion models both take the first option, for exactly this reason.

## compute_loss(): the portable escape hatch

In [ ]:
import keras
from keras import layers, ops
import numpy as np

class WeightedModel(keras.Model):
    """Penalise mistakes on class 0 five times as heavily."""

    def compute_loss(self, x=None, y=None, y_pred=None,
                     sample_weight=None, training=True):
        base = keras.losses.sparse_categorical_crossentropy(y, y_pred)
        weights = ops.where(ops.equal(y, 0), 5.0, 1.0)
        return ops.mean(base * weights)

inputs = keras.Input(shape=(784,))
h = layers.Dense(64, activation="relu")(inputs)
outputs = layers.Dense(10, activation="softmax")(h)
model = WeightedModel(inputs, outputs)

from keras.datasets import mnist
(x, y), (xt, yt) = mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

model.compile(optimizer="rmsprop", metrics=["accuracy"])
model.fit(x, y, epochs=2, batch_size=128, verbose=2)

No `loss=` in `compile()` — the model supplies its own. This runs unchanged on all three backends, because nothing in it touches a gradient.

## train_step(): where the backend leaks in

If you need the gradients themselves — to clip them, to accumulate them, to update two sets of weights alternately — you have to write backend-specific code. Here is the same idea three ways.

In [ ]:
# TensorFlow
class TFModel(keras.Model):
    def train_step(self, data):
        x, y = data
        import tensorflow as tf
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compute_loss(x=x, y=y, y_pred=y_pred)
        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        for m in self.metrics:
            m.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}

In [ ]:
# PyTorch
class TorchModel(keras.Model):
    def train_step(self, data):
        x, y = data
        self.zero_grad()
        y_pred = self(x, training=True)
        loss = self.compute_loss(x=x, y=y, y_pred=y_pred)
        loss.backward()
        trainable = [v for v in self.trainable_weights]
        grads = [v.value.grad for v in trainable]
        with torch.no_grad():
            self.optimizer.apply(grads, trainable)
        for m in self.metrics:
            m.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}

In [ ]:
# JAX -- stateless, so the signature is different entirely
class JaxModel(keras.Model):
    def train_step(self, state, data):
        x, y = data
        (trainable, non_trainable, optimizer_vars, metrics_vars) = state
        grad_fn = jax.value_and_grad(self.compute_loss_and_updates,
                                     has_aux=True)
        (loss, aux), grads = grad_fn(trainable, non_trainable, x, y)
        trainable, optimizer_vars = self.optimizer.stateless_apply(
            optimizer_vars, grads, trainable)
        return logs, (trainable, non_trainable, optimizer_vars, metrics_vars)

Three genuinely different shapes, and the JAX one is not even the same signature. **This is the cost of overriding `train_step`** — and the reason chapter 17 goes out of its way to avoid it.

## A real use: gradient clipping

In [ ]:
# The common case has a built-in, no subclassing required.
model = keras.Sequential([layers.Dense(64, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(
    optimizer=keras.optimizers.RMSprop(clipnorm=1.0),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x, y, epochs=1, batch_size=128, verbose=2)
print("clipnorm handled by the optimizer -- no custom train_step needed")

> **Note** — Before writing a custom `train_step`, check whether the optimizer already takes the argument. `clipnorm`, `clipvalue`, `loss_scale_factor`, `use_ema`, and `weight_decay` all live there, and all of them are things people subclass to reimplement.

## run_eagerly, for when it will not work

In [ ]:
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"],
              run_eagerly=True)     # slow, but print() and pdb work
model.fit(x[:2000], y[:2000], epochs=1, batch_size=128, verbose=2)

Compiled graphs do not run your `print` statements the way you expect and cannot be stepped through. `run_eagerly=True` makes debugging possible and training slow — **turn it on to find the bug, off before measuring anything.**

---

## What to take away

- Override `compute_loss()` when you can — it is portable across all three backends.
- Override `train_step()` only when you need the gradients themselves, and accept that it is backend-specific.
- Check the optimizer's arguments first; clipping, EMA and loss scaling are already there.
- `run_eagerly=True` to debug, off to measure.